Model Choice: GBDT (conform ESP)


# **1. Initialization**

In [6]:
!pip install --quiet gdown rdkit biopython logomaker fair-esm huggingface_hub \
transformers[sentencepiece]

In [8]:
# @title 1.1. Setup & Library Imports

# Standard libraries
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
import random
import numpy as np
import pandas as pd
import math
import re
import shutil
import urllib.request
import sys
import time

# Sequence analysis
from Bio import AlignIO
from Bio.Align import AlignInfo
from Bio.Align import MultipleSeqAlignment
from collections import Counter
from scipy.stats import entropy

# RDKit
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, PandasTools, Draw
RDLogger.DisableLog('rdApp.*') # Suppress RDKit warnings

# Plotting
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, PercentFormatter
import seaborn as sns
import logomaker
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# File downloads
import gdown
from google.colab import drive
#drive.mount('/content/drive')

# Logger setup
import logging
from pathlib import Path
from typing import List, Tuple
import time, logging

# Reproducibility seeds
random.seed(42)
np.random.seed(42)

# Configure module-level logger
# 1) Create module logger and prevent it from propagating to root
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False

# 2) Clear any handlers already attached (just in case)
for h in list(logger.handlers):
    logger.removeHandler(h)

# 3) Attach exactly one StreamHandler to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter(
    "%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
))
logger.addHandler(handler)

# 4) Test it
logger.info("Logger is configured and ready to go!")


# Machine learning
from huggingface_hub import hf_hub_download
from esm import pretrained
from esm.pretrained import _download_model_and_regression_data, \
load_model_and_alphabet_core, load_model_and_alphabet_local
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from scipy.stats import pointbiserialr

# Install minimal, pip-installable dependencies (idempotent).
# Avoid changing Colab's core stack; only add the required packages with exact pins.
import sys
import os, json, random, logging
from pathlib import Path
import numpy as np
import torch

def ensure(pkg_spec: str):
    pkg = pkg_spec.split("==")[0]
    try:
        __import__(pkg.replace("-", "_"))
        print(f"{pkg_spec} -> already installed (keeping existing).")
    except Exception:
        print(f"Installing {pkg_spec} ...")
        # Use !pip install directly for better compatibility in Colab
        !{sys.executable} -m pip install --quiet {pkg_spec}
        # Verify installation
        try:
            __import__(pkg.replace("-", "_"))
            print(f"{pkg_spec} installed successfully.")
        except Exception:
            print(f"Failed to install {pkg_spec}. Please check the output above for details.")


ensure("rdkit")
ensure("xgboost==1.7.6")
ensure("gdown==5.2.0")
print("Environment ready. If packages were just installed, you may optionally restart, but this notebook avoids requiring it.")

ModuleNotFoundError: No module named 'Bio'

In [ ]:
# @title 1.2. Utility Function Definitions

def download_from_drive(file_id, out_path, quiet=True):
    """
    Download a file from Google Drive by share-link ID and save to out_path.
    Ensures the target directory exists and logs progress.
    """
    url = f"https://drive.google.com/uc?id={file_id}"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    logger.info(f"Downloading {file_id} → {out_path}")
    gdown.download(url, str(out_path), quiet=quiet)
    return out_path

def load_reactions_tsv(path, required_cols=("enzyme", "acceptor", "sequence",
                                            "csmiles", "cs_single")):
    """
    Load reaction data from a TSV, verify required columns, and
    add a binary 'reaction' flag (1 if single or double cs present).
    Raises KeyError if any required column is missing.
    """
    df = pd.read_csv(path, sep="\t")
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns in {path}: {missing}")
    # Flag as reactive if either cs_single or cs_double has value
    df["reaction"] = (
        df["cs_single"].notna() | df["cs_double"].notna()
    ).astype(int)
    logger.info(f"Loaded {len(df)} rows with reaction flag.")
    return df

# **2. Data Acquisition**

In [ ]:
# @title 2.1. Download & Load Reaction TSV
tsv_id   = "1dXwZPB5Wbpdp5LjLY73Qj8WWM8ptaY5i" # FILE_ID
tsv_path = Path("data/merged_full.tsv")
download_from_drive(tsv_id, tsv_path, quiet=False)
df = load_reactions_tsv(tsv_path)
df['superclass'].fillna('Other', inplace=True) # Fill NaNs in Superclass
df.head() # Quick inspect

In [ ]:
# Prepare the list of (Enzyme ID, sequence) tuples
unique_df = df.drop_duplicates(subset='enzyme', keep='first')
records   = list(zip(unique_df['enzyme'], unique_df['sequence']))

In [ ]:
# @title 2.2. Download & Load ESM-1b

# Download & cache tokenizer + model
MODEL_NAME      = "facebook/esm1b_t33_650M_UR50S"
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

print(f"Loaded {MODEL_NAME} on {device}")

In [ ]:
# @title 2.3. Generate Morgan fingerprints
# Extract unique substrates
subs = df[['inchikey', 'smiles']].drop_duplicates().reset_index(drop=True)

# Generate radius-2, 1024-bit Morgan FPs
NBITS = 1024
fps = []
for smi in subs.smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        fps.append(np.zeros(NBITS, dtype=int))
    else:
        bv = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=NBITS)
        arr = np.zeros((NBITS,), dtype=int)
        DataStructs.ConvertToNumpyArray(bv, arr)
        fps.append(arr)
fps = np.stack(fps)

# Ensure the output directory exists
os.makedirs('features', exist_ok=True)

# Save substrate features
np.save('features/morgan_fp.npy', fps)
subs.to_csv('features/substrate_index.csv', index=False)

In [ ]:
# @title 2.4. Run Inference for Embeddings

# 1) Parameters
MAX_LEN    = 1000  # only used if TRUNCATE=True
BATCH_SIZE = 32
TRUNCATE   = False  # enable or disable truncation

# 2) Inference loop with progress checks
embs_list, names = [], []
total = len(records)
start_time = time.time()

for batch_idx in range(0, total, BATCH_SIZE):
    batch_num = batch_idx // BATCH_SIZE + 1
    batch = records[batch_idx : batch_idx + BATCH_SIZE]
    ids, seqs = zip(*batch)

    logger.info(
        f"Batch {batch_num}/{math.ceil(total/BATCH_SIZE)}: "
        f"{len(seqs)} seqs, lengths={ [len(s) for s in seqs] }"
    )

    # 2a) Build tokenizer kwargs
    tok_kwargs = dict(return_tensors="pt", padding="longest")
    if TRUNCATE:
        tok_kwargs.update(truncation=True, max_length=MAX_LEN)
    else:
        tok_kwargs["truncation"] = False

    # 2b) Tokenize & transfer
    inputs = tokenizer(list(seqs), **tok_kwargs)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Demoted detailed logs to DEBUG
    logger.debug(f"  tokenized input_ids shape = {inputs['input_ids'].shape}")
    if device.type == "cuda":
        mem_alloc = torch.cuda.memory_allocated(device) / 1e9
        mem_reserved = torch.cuda.memory_reserved(device) / 1e9
        logger.debug(
            f"  GPU before forward: allocated {mem_alloc:.2f}GB, "
            f"reserved {mem_reserved:.2f}GB"
        )

    # 2c) Forward
    with torch.no_grad():
        outputs = model(**inputs)

    if device.type == "cuda":
        mem_alloc2 = torch.cuda.memory_allocated(device) / 1e9
        mem_reserved2 = torch.cuda.memory_reserved(device) / 1e9
        logger.debug(
            f"  GPU after forward: allocated {mem_alloc2:.2f}GB, "
            f"reserved {mem_reserved2:.2f}GB"
        )

    # 2d) Mean-pool
    reps = outputs.last_hidden_state  # [B, L, H]
    mask = inputs["attention_mask"].unsqueeze(-1)
    summed = (reps * mask).sum(dim=1)
    counts = mask.sum(dim=1)
    batch_embs = (summed / counts).cpu().numpy()

    # Verify shape at DEBUG level
    logger.debug(
        f"  embeddings batch shape = {batch_embs.shape} "
        f"(should be ({len(seqs)}, {batch_embs.shape[1]}))"
    )

    embs_list.append(batch_embs)
    names.extend(ids)

    # Small sleep to ensure log flushing
    time.sleep(0.05)

# 3) Stack & save
logger.info("Stacking all batches and saving to disk…")
emb_array = np.vstack(embs_list)
logger.info(f"Final embedding array shape = {emb_array.shape}")

# Save outputs
os.makedirs('features', exist_ok=True)
np.save('features/esm1b_emb.npy', emb_array)
pd.Series(names, name='enzyme').to_csv('features/esm_index.csv', index=False)

elapsed = time.time() - start_time
logger.info(f"Done! Total time = {elapsed:.1f}s")

In [4]:
# Seeds and determinism; device detection for ESM inference; GBDT trains on CPU.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    torch.manual_seed(SEED)
    torch.use_deterministic_algorithms(True)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception as e:
    print("Determinism note:", e)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU (ESM falls back to CPU; GBDT trains on CPU).")

N_JOBS = os.cpu_count() or 2
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


Running on CPU (ESM falls back to CPU; GBDT trains on CPU).


In [2]:
# Init & seeds
import os, json, random, logging
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
try:
    torch.manual_seed(SEED)
    torch.use_deterministic_algorithms(True)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception as e:
    print(f"Determinism setup note: {e}")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print('GPU available:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU (GBDT trains on CPU; ESM inference will fallback to CPU).')

N_JOBS = os.cpu_count() or 2
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')


Running on CPU (GBDT trains on CPU; ESM inference will fallback to CPU).


In [3]:
# Project root (Drive of lokaal) & mappen
USE_DRIVE = True  # zet op False voor volledige herbouw per run zonder persistentie
PROJ_NAME = 'esi_baseline_gbdt'
if USE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
        PROJ = Path('/content/drive/MyDrive')/PROJ_NAME
    except Exception as e:
        print('Drive mount failed; falling back to local /content (ephemeral).', e)
        PROJ = Path('/content')/PROJ_NAME
else:
    PROJ = Path('/content')/PROJ_NAME

for sub in ['data','features','splits','models','metrics','logs','reports']:
    (PROJ/sub).mkdir(parents=True, exist_ok=True)
print('PROJ =', PROJ)


Drive mount failed; falling back to local /content (ephemeral). Error: credential propagation was unsuccessful
PROJ = /content/esi_baseline_gbdt


In [4]:
# Build artifacts (2.1–2.4): reactions.tsv, ESM-1b embeddings, Morgan FPs, scaffold-splits
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from transformers import AutoTokenizer, AutoModel
import gdown

DATA_TSV = PROJ/'data'/'reactions.tsv'
FEAT_DIR = PROJ/'features'
SPLITS_JSON = PROJ/'splits'/'scaffold_folds.json'

# ---- 2.1 Download & Load Reaction TSV ----
if not DATA_TSV.exists():
    file_id = os.environ.get('REACTIONS_TSV_GDRIVE_ID', '').strip()
    if file_id:
        url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(url, str(DATA_TSV), quiet=False)
    else:
        # Minimal fallback: instruct user to upload; keeps notebook self-voorzienend in Colab UI
        raise FileNotFoundError('Missing PROJ/data/reactions.tsv. Set env REACTIONS_TSV_GDRIVE_ID or upload reactions.tsv to PROJ/data/.')

df = pd.read_csv(DATA_TSV, sep='\t' if DATA_TSV.suffix=='.tsv' else ',', dtype=str).fillna('')
required_cols = ['enzyme','sequence','csmiles']
for c in required_cols:
    assert c in df.columns, f"Missing required column: {c}"
# label: reaction = 1 if cs_single or cs_double present; otherwise 0
if 'reaction' not in df.columns:
    cs_single = df.get('cs_single','')
    cs_double = df.get('cs_double','')
    df['reaction'] = ((cs_single!='') | (cs_double!='')).astype(int)

# Canonicalize SMILES
def canon_smiles(smi: str) -> str:
    try:
        m = Chem.MolFromSmiles(smi)
        return Chem.MolToSmiles(m, canonical=True) if m else ''
    except Exception:
        return ''

df['csmiles'] = df['csmiles'].apply(canon_smiles)
df = df[df['csmiles']!=''].reset_index(drop=True)

# ---- 2.3 Generate Morgan fingerprints ----
RADIUS = 2
NBITS = 1024
subs = sorted(df['csmiles'].unique())
sub_to_idx = {s:i for i,s in enumerate(subs)}
fps = np.zeros((len(subs), NBITS), dtype=np.uint8)
bad = 0
for s,i in sub_to_idx.items():
    m = Chem.MolFromSmiles(s)
    if m is None:
        bad += 1
        continue
    bv = AllChem.GetMorganFingerprintAsBitVect(m, RADIUS, nBits=NBITS, useChirality=False, useBondTypes=True)
    arr = np.zeros(NBITS, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(bv, arr)
    fps[i] = arr
np.save(FEAT_DIR/'morgan_fp.npy', fps)
pd.DataFrame({'sub_idx': list(range(len(subs))), 'smiles': subs}).to_csv(FEAT_DIR/'sub_index.csv', index=False)
print(f"Morgan FPs: {fps.shape}, invalid smiles: {bad}")

# ---- 2.2 & 2.4 Run Inference for ESM-1b embeddings ----
enz = sorted(df['sequence'].unique())
enz_to_idx = {s:i for i,s in enumerate(enz)}
EMB_NPY = FEAT_DIR/'esm1b_emb.npy'
if not EMB_NPY.exists():
    MODEL_NAME = 'facebook/esm1b_t33_650M_UR50S'
    HF_REV = os.environ.get('ESM1B_REV', '7b37824baec4d3658e1df7479222a7c79b465b76')  # pinned
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, revision=HF_REV)
    mdl = AutoModel.from_pretrained(MODEL_NAME, revision=HF_REV).to(DEVICE)
    mdl.eval()
    BATCH_SIZE = 8
    embs = np.zeros((len(enz), 1280), dtype=np.float32)
    i = 0
    local_device = DEVICE
    with torch.no_grad():
        while i < len(enz):
            bs = min(BATCH_SIZE, len(enz) - i)
            while True:
                try:
                    batch = enz[i:i+bs]
                    enc = tok(batch, return_tensors='pt', padding=True, truncation=True)
                    enc = {k: v.to(local_device) for k,v in enc.items()}
                    out = mdl(**enc).last_hidden_state  # [B, L, 1280]
                    assert out.size(-1) == 1280, f"Unexpected hidden size: {out.size(-1)}"
                    mask = enc['attention_mask'].unsqueeze(-1)
                    pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                    embs[i:i+bs, :] = pooled.detach().cpu().numpy()
                    i += bs
                    break
                except RuntimeError as e:
                    if 'out of memory' in str(e).lower() and local_device == 'cuda':
                        torch.cuda.empty_cache()
                        if bs > 1:
                            bs = max(1, bs//2)
                            print(f"[ESM] CUDA OOM — reducing batch size to {bs}")
                            continue
                        else:
                            print('[ESM] CUDA OOM at batch size 1 — falling back to CPU')
                            mdl = mdl.to('cpu')
                            local_device = 'cpu'
                            continue
                    raise
    np.save(EMB_NPY, embs)
pd.DataFrame({'enz_idx': list(range(len(enz))), 'sequence': enz}).to_csv(FEAT_DIR/'enz_index.csv', index=False)
print('ESM embeddings ready:', np.load(EMB_NPY).shape)

# ---- Generate scaffold-based 5-fold CV if missing ----
if not SPLITS_JSON.exists():
    # Build pair index
    df_pairs = df[['sequence','csmiles','reaction']].copy()
    df_pairs['enz_idx'] = df_pairs['sequence'].map(enz_to_idx)
    df_pairs['sub_idx'] = df_pairs['csmiles'].map(sub_to_idx)
    def scaffold(smi):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            return ''
        scaf = MurckoScaffold.GetScaffoldForMol(m)
        return Chem.MolToSmiles(scaf) if scaf else ''
    df_pairs['scaffold'] = df_pairs['csmiles'].apply(scaffold)

    # Assign scaffolds to folds round-robin, stratified by label frequency per scaffold
    scaf_groups = df_pairs.groupby('scaffold')
    scafs = list(scaf_groups.groups.keys())
    scafs.sort(key=lambda s: -scaf_groups.get_group(s)['reaction'].sum() if s in scaf_groups.groups else 0)
    K = 5
    folds = {k: {'train': [], 'valid': []} for k in range(K)}
    scaf_to_fold = {s: (i % K) for i, s in enumerate(scafs)}
    for idx, row in df_pairs.iterrows():
        k = scaf_to_fold.get(row['scaffold'], idx % K)
        folds[k]['valid'].append(int(idx))
    all_ids = set(range(len(df_pairs)))
    for k in range(K):
        val_ids = set(folds[k]['valid'])
        folds[k]['train'] = sorted(list(all_ids - val_ids))
        folds[k]['valid'] = sorted(list(val_ids))
    with open(SPLITS_JSON, 'w') as f:
        json.dump(folds, f)
    print('Scaffold CV created:', SPLITS_JSON)
else:
    print('Using existing splits:', SPLITS_JSON)


ModuleNotFoundError: No module named 'rdkit'

In [ ]:
# CV training (GBDT) met grid: leest splits; schrijft metrics/models
from itertools import product
from sklearn.metrics import roc_auc_score, accuracy_score, matthews_corrcoef, brier_score_loss
from xgboost import XGBClassifier

DF = pd.read_csv(PROJ/'data'/'reactions.tsv', sep='\t' if (PROJ/'data'/'reactions.tsv').suffix=='.tsv' else ',').fillna('')
if 'reaction' not in DF.columns:
    DF['reaction'] = ((DF.get('cs_single','')!='') | (DF.get('cs_double','')!='')).astype(int)
DF = DF[DF['csmiles']!=''].reset_index(drop=True)

enz_index = pd.read_csv(PROJ/'features'/'enz_index.csv')
sub_index = pd.read_csv(PROJ/'features'/'sub_index.csv')
enz_to_idx = dict(zip(enz_index['sequence'], enz_index['enz_idx']))
sub_to_idx = dict(zip(sub_index['smiles'], sub_index['sub_idx']))
embs = np.load(PROJ/'features'/'esm1b_emb.npy')
fps  = np.load(PROJ/'features'/'morgan_fp.npy')

pairs = DF[['sequence','csmiles','reaction']].copy()
pairs['enz_idx'] = pairs['sequence'].map(enz_to_idx)
pairs['sub_idx'] = pairs['csmiles'].map(sub_to_idx)
pairs = pairs.dropna(subset=['enz_idx','sub_idx']).reset_index(drop=True)

with open(PROJ/'splits'/'scaffold_folds.json','r') as f:
    folds = json.load(f)

grid = {
  'learning_rate': [0.01, 0.05],
  'max_depth': [6, 10],
  'n_estimators': [400, 800],
  'subsample': [0.7, 1.0]
}
grid_list = list(product(grid['learning_rate'], grid['max_depth'], grid['n_estimators'], grid['subsample']))

for k_str, fold in folds.items():
    k = int(k_str)
    fold_dir = PROJ/'models'/'gbdt'/f'fold{k}'
    (fold_dir).mkdir(parents=True, exist_ok=True)
    mdir = PROJ/'metrics'/'cv'/'gbdt'
    mdir.mkdir(parents=True, exist_ok=True)

    tr_ids = fold['train']
    va_ids = fold['valid']
    tr = pairs.iloc[tr_ids]
    va = pairs.iloc[va_ids]

    X_tr = np.hstack([embs[tr['enz_idx'].astype(int)], fps[tr['sub_idx'].astype(int)]])
    y_tr = tr['reaction'].astype(int).values
    X_va = np.hstack([embs[va['enz_idx'].astype(int)], fps[va['sub_idx'].astype(int)]])
    y_va = va['reaction'].astype(int).values

    # handle imbalance
    pos = (y_tr==1).sum()
    neg = (y_tr==0).sum()
    scale_pos = max(1.0, neg / max(1, pos))

    best = {'roc_auc': -1, 'params': None, 'model': None}
    for lr, md, ne, ss in grid_list:
        clf = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            learning_rate=lr, max_depth=md, n_estimators=ne, subsample=ss,
            scale_pos_weight=scale_pos, n_jobs=N_JOBS, random_state=SEED
        )
        clf.fit(X_tr, y_tr)
        p_va = clf.predict_proba(X_va)[:,1]
        yhat = (p_va >= 0.5).astype(int)
        roc = roc_auc_score(y_va, p_va)
        acc = accuracy_score(y_va, yhat)
        mcc = matthews_corrcoef(y_va, yhat)
        bri = brier_score_loss(y_va, p_va)
        if roc > best['roc_auc']:
            best = {
                'roc_auc': roc, 'acc': acc, 'mcc': mcc, 'brier': bri,
                'params': {'learning_rate': lr, 'max_depth': md, 'n_estimators': ne, 'subsample': ss},
                'model': clf
            }
    # save best model and metrics
    best['model'].get_booster().save_model(str(fold_dir/'model.bin'))
    with open(mdir/f'fold{k}.json','w') as f:
        out = {'fold': k, 'metrics': {k2: float(v) for k2,v in best.items() if k2 in ['roc_auc','acc','mcc','brier']}, 'best_params': best['params']}
        json.dump(out, f, indent=2)
    print(f"Fold {k} | ROC-AUC={best['roc_auc']:.3f} ACC={best['acc']:.3f} MCC={best['mcc']:.3f}")


In [ ]:
# Aggregatie & rapportage
from glob import glob

mdir = PROJ/'metrics'/'cv'/'gbdt'
fold_files = sorted(glob(str(mdir/'fold*.json')))
rows = []
for fp in fold_files:
    with open(fp,'r') as f:
        rows.append(json.load(f))
if rows:
    dfm = pd.json_normalize(rows)
    df_sum = pd.DataFrame({
        'metric': ['roc_auc','acc','mcc','brier'],
        'mean': [dfm['metrics.roc_auc'].mean(), dfm['metrics.acc'].mean(), dfm['metrics.mcc'].mean(), dfm['metrics.brier'].mean()],
        'std':  [dfm['metrics.roc_auc'].std(), dfm['metrics.acc'].std(), dfm['metrics.mcc'].std(), dfm['metrics.brier'].std()],
    })
    df_sum.to_csv(PROJ/'metrics'/'cv'/'gbdt'/'summary.csv', index=False)
    # simple markdown report
    grid_line = 'learning_rate∈{0.01,0.05}; max_depth∈{6,10}; n_estimators∈{400,800}; subsample∈{0.7,1.0}'
    report = [
        '# Baseline CV Report — GBDT (conform ESP)\n',
        f'- Splits: {PROJ}/splits/scaffold_folds.json\n',
        f'- Hyperparameter grid: {grid_line}\n',
        '\n',
        '## Summary metrics (mean ± std)\n'
    ]
    for _,r in df_sum.iterrows():
        report.append(f"- {r['metric']}: {r['mean']:.4f} ± {r['std']:.4f}\n")
    (PROJ/'reports'/'baseline_cv_report.md').write_text(''.join(report))
    print('Wrote summary and report.')
else:
    print('No fold metrics found to aggregate.')


## Deliverables

| WBS-stap | Afhankelijke stap(pen) | Notebookbron | Uit te voeren taak | Verwachte output |
|---|---|---|---|---|
| 5.1 | 1. Initialization; 2.1–2.4 | 1.1, 1.2, 2.1–2.4 | Zet seeds (SEED=42; random/numpy/torch) en deterministische flags; device-detectie (GPU voor ESM inference indien beschikbaar; GBDT op CPU); kies persistentie: Drive-mount onder `PROJ` of herbouw per run. | Logregels; mapstructuur `PROJ/{data,features,splits,models,metrics,logs,reports}` |
| 5.1.1 | 2.1, 2.2, 2.3, 2.4 | 2.1, 2.2, 2.3, 2.4 | Voorbereid trainingsscripts: laad `features/esm1b_emb.npy` + `features/enz_index.csv` en `features/morgan_fp.npy` + `features/sub_index.csv`; join op `(sequence,csmiles)`; logs naar `logs/`. | `features/esm1b_emb.npy`, `features/enz_index.csv`, `features/morgan_fp.npy`, `features/sub_index.csv` |
| 5.1.2 | 5.1.2.1; 5.1.2.2 | 2.4 (splits) | Cross-validation over scaffold-splits; splits aanwezig als `splits/scaffold_folds.json` (fold→{train,valid}); indien afwezig: **N/A + minimale toevoeging** (genereer scaffold-splits). | `models/gbdt/fold{K}/model.bin`, `metrics/cv/gbdt/fold{K}.json` |
| 5.1.2.1 | 2.4 | 2.4 | Schrijf CV-trainingsscript met hyperparametergrid **GBDT**: “learning_rate∈{0.01,0.05}; max_depth∈{6,10}; n_estimators∈{400,800}; subsample∈{0.7,1.0}”; gebruik `scale_pos_weight`. | Per fold beste params in `metrics/cv/gbdt/fold{K}.json` |
| 5.1.2.2 | 5.1.2.1 | N/A | Voer CV-runs lokaal (sequentieel/parallel via `n_jobs`); GBDT training op CPU (`tree_method='hist'`); ESM inference op GPU indien beschikbaar. | `models/gbdt/fold{K}/model.bin`, logs |
| 5.1.2.3 | 5.1.2 | N/A | Agregeer foldresultaten en bereken ROC-AUC, ACC, MCC, Brier; schrijf samenvatting (mean±std). | `metrics/cv/gbdt/summary.csv` |
| 5.1.2.4 | 5.1.2.3 | N/A | Documenteer CV-proces en uitkomsten (grid, splits, metrics) in Markdown. | `reports/baseline_cv_report.md` |
| 5.1.3 | 5.1.2 | N/A | Registreer prestaties en bewaar getrainde artefacten per fold; vaste paden onder `models/gbdt/fold{K}/` en `metrics/cv/gbdt/`. | `models/gbdt/fold{K}/model.bin`, `metrics/cv/gbdt/fold{K}.json` |

### Checklist (max. 8 stappen)
1. Mount Drive of kies herbouw per run en maak `PROJ`-mappen aan.
2. Pin dependencies en **restart runtime** indien nodig.
3. Init seeds/determinisme en device-detectie (GPU alleen voor ESM inference).
4. Bouw/valideer artifacts: `reactions.tsv` → `features/esm1b_emb.npy` & `features/morgan_fp.npy` (+ index CSV’s).
5. Genereer of laad `splits/scaffold_folds.json` (fold→{train,valid}).
6. Train **GBDT** per fold over grid en bewaar `model.bin` + fold-metrics JSON.
7. Agregeer metrics en schrijf `metrics/cv/gbdt/summary.csv`.
8. Genereer `reports/baseline_cv_report.md` met grid, splits en samenvatting.
